In [1]:
import pandas as pd
import numpy as np

In [2]:
main_df = pd.read_csv("chiller_predictive_maintenance_dataset.csv")

In [3]:
train_df = main_df[main_df["Split"] == "train"].copy()

holdout_df = main_df[main_df["Split"] == "holdout_manual"].copy()

In [4]:
input_cols = [
    "F36", "F15", "H25", "H24", "H64", "H15", "F14", "H42",
    "Mode_1Cool_2Heat", "H23F", "H23C", "P02", "H20", "P03",
    "F37", "F39", "H62"
]

output_cols = [
    "Chiller_Condenser_Fault",
    "Chiller_ExpansionValve_Fault",
    "Chiller_Evaporator_Fault",
    "Chiller_Compressor_Fault",
    "Chiller_General_Fault",
    "Condenser_Valve_Fault",
    "Condenser_Filter_Fault",
    "Condenser_GasChargeNeeded",
    "Condenser_Fan_Fault",
    "Condenser_Coil_Fault",
    "Condenser_GasLeak",
    "Evaporator_LiquidLeak",
    "Evaporator_Valve_Fault",
    "Evaporator_StrainerFilter_Fault",
    "Evaporator_Piping_Fault",
    "Evaporator_HeatExchanger_Fault",
    "Evaporator_Electrical_Fault",
    "Evaporator_FlowSwitch_Fault",
    "Compressor_General_Fault",
    "Compressor_Contactor_Fault",
    "Compressor_Leak",
    "Compressor_OilLoss",
    "Compressor_SuperheatIncreaseNeeded",
    "General_Electrical_Fault",
    "General_LowPerformance",
    "General_UnitUndersizedOversized"
]

In [5]:
X = train_df[input_cols]
Y = train_df[output_cols]

In [6]:
test_size=0.2
random_state=42

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [9]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

mlp = keras.Sequential([
    layers.Input(shape=(17,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(26, activation="linear")
])

In [10]:
mlp.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [11]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = mlp.fit(
    X_train_scaled,
    Y_train,
    validation_data=(X_val_scaled, Y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0669 - mae: 0.1945 - val_loss: 0.0333 - val_mae: 0.1424
Epoch 2/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0300 - mae: 0.1345 - val_loss: 0.0276 - val_mae: 0.1280
Epoch 3/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0256 - mae: 0.1226 - val_loss: 0.0240 - val_mae: 0.1170
Epoch 4/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0225 - mae: 0.1128 - val_loss: 0.0213 - val_mae: 0.1084
Epoch 5/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0203 - mae: 0.1055 - val_loss: 0.0194 - val_mae: 0.1026
Epoch 6/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0186 - mae: 0.0999 - val_loss: 0.0179 - val_mae: 0.0974
Epoch 7/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0173 - mae: 0.0950 - val_loss: 0.0168 - val_mae: 0.0931
Epoch 8/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0162 - mae: 0.0914 - val_loss: 0.0159 - val_mae: 0.0901
Epoch 9/100
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/

In [12]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

history_2 = mlp.fit(
    X_train_scaled,
    Y_train,
    validation_data=(X_val_scaled, Y_val),
    initial_epoch=100,
    epochs=200,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

Epoch 101/200
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0054 - mae: 0.0434 - val_loss: 0.0059 - val_mae: 0.0446
Epoch 102/200
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0054 - mae: 0.0433 - val_loss: 0.0058 - val_mae: 0.0444
Epoch 103/200
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0053 - mae: 0.0430 - val_loss: 0.0059 - val_mae: 0.0445
Epoch 104/200
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0054 - mae: 0.0432 - val_loss: 0.0059 - val_mae: 0.0447
Epoch 105/200
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0053 - mae: 0.0430 - val_loss: 0.0058 - val_mae: 0.0443
Epoch 106/200
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0053 - mae: 0.0428 - val_loss: 0.0059 - val_mae: 0.0444
Epoch 107/200
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0053 - mae: 0.0428 - val_loss: 0.0058 - val_mae: 0.0446
Epoch 108/200
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0053 - mae: 0.0429 - val_loss: 0.0058 - val_mae: 0.0446
Epoch 109/200
150/150 ━━━━━━━━━━

In [13]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

Y_val_pred_mlp = mlp.predict(X_val_scaled)

mlp_mae = mean_absolute_error(Y_val, Y_val_pred_mlp)
mlp_rmse = np.sqrt(mean_squared_error(Y_val, Y_val_pred_mlp))
mlp_r2 = r2_score(Y_val, Y_val_pred_mlp, multioutput="uniform_average")

print("MLP Validation R²:", mlp_r2)
print("MLP Validation MAE:", mlp_mae)
print("MLP Validation RMSE:", mlp_rmse)

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
MLP Validation R²: 0.8228176832199097
MLP Validation MAE: 0.0415760837495327
MLP Validation RMSE: 0.07261280483668468


In [14]:
from sklearn.metrics import r2_score
import pandas as pd

mlp_r2_per_output = r2_score(
    Y_val,
    Y_val_pred_mlp,
    multioutput="raw_values"
)

mlp_r2_df = pd.DataFrame({
    "Output": output_cols,
    "R2": mlp_r2_per_output
})

mlp_r2_df = mlp_r2_df.sort_values("R2", ascending=False)

print(mlp_r2_df.to_string(index=False))

                            Output       R2
       Evaporator_FlowSwitch_Fault 0.949227
      Chiller_ExpansionValve_Fault 0.947307
    Evaporator_HeatExchanger_Fault 0.942305
             Condenser_Valve_Fault 0.931631
            Condenser_Filter_Fault 0.931631
   Evaporator_StrainerFilter_Fault 0.926454
         Condenser_GasChargeNeeded 0.924851
           Evaporator_Piping_Fault 0.905722
              Condenser_Coil_Fault 0.898066
                 Condenser_GasLeak 0.898066
             Evaporator_LiquidLeak 0.859075
            Evaporator_Valve_Fault 0.859074
               Condenser_Fan_Fault 0.843919
          Chiller_Evaporator_Fault 0.837455
Compressor_SuperheatIncreaseNeeded 0.831372
           Chiller_Condenser_Fault 0.824831
                Compressor_OilLoss 0.808343
        Compressor_Contactor_Fault 0.808343
                   Compressor_Leak 0.803552
            General_LowPerformance 0.801318
             Chiller_General_Fault 0.748343
          Chiller_Compressor_Fau

# **Final MLP Training (with 12000 rows)**

In [15]:
X_full = train_df[input_cols]
Y_full = train_df[output_cols]

In [16]:
from sklearn.preprocessing import StandardScaler

final_scaler = StandardScaler()

X_full_scaled = final_scaler.fit_transform(X_full)

In [17]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

final_mlp = keras.Sequential([
    layers.Input(shape=(17,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(26, activation="linear")
])

final_mlp.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [18]:
final_mlp.fit(
    X_full_scaled,
    Y_full,
    epochs=180,
    batch_size=64,
    verbose=1
)

Epoch 1/180
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.0636 - mae: 0.1864
Epoch 2/180
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0275 - mae: 0.1284
Epoch 3/180
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0238 - mae: 0.1177
Epoch 4/180
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0206 - mae: 0.1074
Epoch 5/180
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0182 - mae: 0.0989
Epoch 6/180
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0167 - mae: 0.0937
Epoch 7/180
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0157 - mae: 0.0900
Epoch 8/180
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0149 - mae: 0.0872
Epoch 9/180
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0143 - mae: 0.0849
Epoch 10/180
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0138 - mae: 0.0831
Epoch 11/180
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0134 - mae: 0.0816
Epoch 12/180
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0131 - mae: 0.0803
Epoch 13/180


In [19]:
X_holdout = holdout_df[input_cols]
Y_holdout = holdout_df[output_cols]

In [20]:
X_holdout_scaled = final_scaler.transform(X_holdout)

In [21]:
Y_holdout_pred_mlp = final_mlp.predict(X_holdout_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step


In [22]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

holdout_mlp_mae = mean_absolute_error(Y_holdout, Y_holdout_pred_mlp)

holdout_mlp_rmse = np.sqrt(
    mean_squared_error(Y_holdout, Y_holdout_pred_mlp)
)

holdout_mlp_r2 = r2_score(
    Y_holdout,
    Y_holdout_pred_mlp,
    multioutput="uniform_average"
)

print("MLP Holdout R²:", holdout_mlp_r2)
print("MLP Holdout MAE:", holdout_mlp_mae)
print("MLP Holdout RMSE:", holdout_mlp_rmse)

MLP Holdout R²: 0.7043291330337524
MLP Holdout MAE: 0.056781794875860214
MLP Holdout RMSE: 0.10719953849602176


In [23]:

import os
import json
import joblib


os.makedirs("model_exports", exist_ok=True)

final_mlp.save("model_exports/mlp_model.keras")

joblib.dump(final_scaler, "model_exports/input_scaler.pkl")

metadata = {
    "input_columns": list(input_cols),
    "output_columns": list(output_cols),
    "model_type": "MLP multi-output regression",
    "input_scaling": "StandardScaler",
    "training_rows": 12000,
    "epochs": 180,
    "batch_size": 64
}

with open("model_exports/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Export completed:")
print(os.listdir("model_exports"))


Export completed:
['input_scaler.pkl', 'mlp_model.keras', 'model_metadata.json']


In [25]:
import shutil

shutil.make_archive(
    "model_exports",
    "zip",
    "model_exports"
)

'E:\\predictive\\model_exports.zip'

In [26]:

import numpy as np
import joblib
import tensorflow as tf

loaded_mlp = tf.keras.models.load_model(
    "model_exports/mlp_model.keras"
)

loaded_scaler = joblib.load(
    "model_exports/input_scaler.pkl"
)

sample = X_full.iloc[[0]]
sample_scaled = loaded_scaler.transform(sample)

prediction = loaded_mlp.predict(sample_scaled, verbose=0)

print("Prediction shape:", prediction.shape)
print("Number of outputs:", prediction.shape[1])
print("All predictions finite:", np.isfinite(prediction).all())


Prediction shape: (1, 26)
Number of outputs: 26
All predictions finite: True
